### 기본설정

 <br>
크롬 버전에 맞춰서 웹브라우저를 다운 받아야 합니다.

크롬 버전 확인하는 방법입니다. 
크롬을 열어준후 설정에서 설치된 크롬의 버전의 확인합니다.

![크롬버전 확인하기1](https://github.com/jeong-wooseok/Python/blob/master/img/img_20201227_5.png?raw=true)

![크롬버전 확인하기2](https://github.com/jeong-wooseok/Python/blob/master/img/img_20201227_6.png?raw=true)

[크롬드라이버다운로드 바로가기](https://chromedriver.chromium.org/downloads)
<br>
소수점 앞의 두자리가 동일한 것으로 다운로드 합니다.

![크롬버전 확인하기2](https://github.com/jeong-wooseok/Python/blob/master/img/img_20201227_7.png?raw=true)


현재 파일 경로를 확인하고, 크롬 드라이버를 넣어줍니다.

In [1]:
pwd

'C:\\Users\\Administrator'

### 크롤링을 통해 ETF 데이터 가져오기

파이썬의 라이브러리인 셀레니움을 사용하여 크롤링을 진행합니다. 
한국예탁결제원의 증권정보포탈 사이트를 참조합니다.

In [57]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time
import pandas as pd
import os

try:
    # 홈페이지에 접근하여 ETF 항목 클릭
    driver=webdriver.Chrome("./chromedriver.exe")
    driver.get("http://seibro.or.kr/websquare/control.jsp?w2xPath=/IPORTAL/user/etf/BIP_CNTS06025V.xml&menuNo=174")
    time.sleep(5)
    a=driver.find_element_by_xpath('//*[@id="image10"]')
    a.click()
    path = 'ETF.txt'
    if os.path.exists(path):os.remove(path)

    # 페이지 넘김
    i=1

    while True: 
        time.sleep(1)
        a=driver.find_element_by_xpath('//*[@id="pageList1_page_%d"]'%i)
        a.click()

        time.sleep(2)
        html=driver.page_source
        soup=BeautifulSoup(html,"html.parser")

        f = open("ETF.txt", 'a')
        for anchors in soup.select("#grid1_body_tbody")[0]:
            for anchor in anchors.select("td"):
                data = anchor.get_text()+'\n'
                f.write(data)
        f.close()
        i+=1
    
    
# 더이상 읽을 테이지 없을 때, 
except:
    # 리스트화 하여 항목을 담아주고, 딕셔너리로 변환하여, 판다스 DF만들기
    종목명 = [] #1;
    코드 = [] #2
    유형 = [] #3
    순자산 =[] #4
    종가 =[] #5
    거래량 = [] #6
    수익율 = [] #9
    총보수 = [] #15
    운용사 = [] #16
    with open('ETF.txt','r') as f:
        data = f.readlines()
        for i, j in enumerate(data):
            if i % 17 == 1:종목명.append(j.replace('\n',''))
            elif i % 17 ==2:코드.append(j.replace('\n',''))
            elif i % 17 ==3:유형.append(j.replace('\n',''))
            elif i % 17 ==4:순자산.append(j.replace('\n',''))
            elif i % 17 ==5:종가.append(j.replace('\n',''))
            elif i % 17 ==6:거래량.append(j.replace('\n',''))
            elif i % 17 ==9:수익율.append(j.replace('\n',''))
            elif i % 17 ==15:총보수.append(j.replace('\n',''))
            elif i % 17 ==16:운용사.append(j.replace('\n',''))      
    data = {'종목명':종목명,'코드':코드,'유형':유형,'순자산':순자산,'종가':종가,'거래량':거래량,'수익율':수익율,'총보수':총보수,'운용사':운용사}
    df = pd.DataFrame(data, columns=['종목명','코드','유형','순자산','종가','거래량','수익율','총보수','운용사'])

    # 보수, 수익율 없는 상품 제거하기
    df = df.drop(df[df['총보수']==''].index)
    df = df.drop(df[df['수익율']==''].index)

    # 전처리 : 계산 가능한 값으로 변환하기
    df['거래량']=df['거래량'].apply(lambda x: x.replace(',',''))
    df['종가']=df['종가'].apply(lambda x: x.replace(',',''))
    df['순자산']=df['순자산'].apply(lambda x: x.replace(',',''))
    df['거래량']=df['거래량'].astype(int)
    df['종가']=df['종가'].astype(int)
    df['순자산']=df['순자산'].astype(int)
    df['수익율']=df['수익율'].astype(float)
    df['총보수']=df['총보수'].astype(float)

    # 전처리 : 5개 자리인 코드를 6개자리로 변환
    df['코드'] = df['코드'].astype(str)
    code = [i for i in df[df['코드'].apply(lambda x: len(x))<=5].index]
    df.loc[code,'코드'] = '0'+ df.loc[code,'코드'].astype(str)

    df.to_csv('./ETF.csv',sep=',',encoding='euc-kr')
    
    print('연산이 끝났습니다')
    print(f'추출한 총 ETF의 수 {len(df)}개')

연산이 끝났습니다
추출한 총 ETF의 수 450개


### 파생상품 제외한 목록 만들기 (연금계좌 활용)

In [50]:
a = ['파생상품/레버리지','파생상품/인버스']
df1 = df[~df['유형'].isin(a)]
df1['유형'].unique()

array(['시장지수/코스피200', '채권/단기자금', '시장지수/MSCI Korea Index', '채권/회사채권',
       '그룹주/삼성', '해외지수/해외지수', '상품/원유', '시장지수/코스닥', '테마/우량주', '섹터/IT',
       '테마/기타', '스타일/스타일', '상품/금속', '채권/국공채권', '섹터/금융', '섹터/경기소비재',
       '테마/SRI', '섹터/의료', '시장지수/KRX300', '기타/기타', '통화/미국달러', '상품/농산물',
       '투자전략/투자전략', '섹터/소재', '그룹주/현대차', '섹터/산업재', '파생상품/구조화', '섹터/필수소비재',
       '시장지수/코스피50', '그룹주/기타', '시장지수/코스피100', '시장지수/KRX100', '통화/기타통화',
       '그룹주/LG', '섹터/통신서비스'], dtype=object)

#### 연금계좌 활용용 ETF : 3개월 수익율 순위

In [51]:
df1.to_csv('./ETF_IRP.csv',sep=',',encoding='euc-kr')
df1.sort_values('수익율',ascending =False)[:10]

,종목명,코드,유형,순자산,종가,거래량,수익율,총보수,운용사
196,KODEX 미국러셀2000(H),280930,해외지수/해외지수,189,13125,18416,38.67,0.45,삼성자산운용
100,KBSTAR 미국S&P원유생산기업(합성 H),219390,해외지수/해외지수,606,2615,581464,35.49,0.25,케이비자산운용
34,KODEX 2차전지산업,305720,테마/기타,3317,15040,1600151,33.57,0.45,삼성자산운용
329,TIGER 라틴35,105010,해외지수/해외지수,85,3245,40713,33.54,0.49,미래에셋자산운용
388,KODEX 운송,140710,섹터/산업재,64,4550,104667,33.24,0.45,삼성자산운용
193,KODEX 미국S&P바이오(합성),185680,해외지수/해외지수,191,36840,1879,32.02,0.25,삼성자산운용
26,TIGER 200 IT,139260,섹터/IT,4258,33255,381483,30.23,0.40,미래에셋자산운용
72,KINDEX 삼성그룹섹터가중,108450,그룹주/삼성,1074,15100,21457,30.17,0.15,한국투자신탁운용
320,KODEX 철강,117680,섹터/소재,89,7025,48240,30.09,0.45,삼성자산운용
152,TIGER 200 에너지화학,139250,섹터/소재,304,17235,57801,29.98,0.40,미래에셋자산운용


### 금일 ETF 수익율 순위 확인

In [1]:
import requests
import json
from pandas import json_normalize
url = 'https://finance.naver.com/api/sise/etfItemList.nhn'
json_data = json.loads(requests.get(url).text)
df2 = json_normalize(json_data['result']['etfItemList'])
df2.sort_values('changeRate',ascending=False)[:10]
df2.columns = ['code','TabCode','이름','현재가','등/락','가격변동','일수익율','일평균가','3개월수익율','거래량/일','거래금액/일,백만','marketSum']
df2 = df2.fillna(0)
df2['3개월수익율'] = df2['3개월수익율'].round(1)
df3 = df2[['code','일수익율','가격변동','거래금액/일,백만']]
df3['코드'] = df3['code']
df2.sort_values('일수익율',ascending=False)[:50]
# df.to_excel('etf_list.xlsx',header=True,index=False);

<ipython-input-1-a3e00e12d0e1>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['코드'] = df3['code']


,code,TabCode,이름,현재가,등/락,가격변동,일수익율,일평균가,3개월수익율,거래량/일,"거래금액/일,백만",marketSum
286,371130,4,KINDEX 블룸버그베트남VN30선물레버리지(H),12065,2,450,3.87,0.0,0.0,28985,351,109
125,243880,2,TIGER 200IT레버리지,52280,2,1785,3.54,51990.0,67.2,10864,562,392
74,267770,3,TIGER 200선물레버리지,16640,2,450,2.78,16635.0,55.4,1242474,20596,1032
171,253250,3,KOSEF 200선물레버리지,19945,2,515,2.65,19934.0,55.5,18884,373,259
6,122630,3,KODEX 레버리지,22100,2,555,2.58,22281.0,54.8,11404382,251171,14984
45,278240,1,KBSTAR 코스닥150선물레버리지,16265,2,400,2.52,16262.0,39.3,23995,386,2232
109,371460,4,TIGER 차이나전기차SOLACTIVE,11895,2,290,2.50,0.0,0.0,63706,755,519
164,123320,3,TIGER 레버리지,20915,2,510,2.50,20939.0,56.2,239724,4997,272
60,233160,1,TIGER 코스닥150 레버리지,17245,2,415,2.47,17309.0,38.3,188625,3221,1423
203,306950,3,KODEX KRX300레버리지,17650,2,415,2.41,17707.0,52.8,10557,186,177
